In [39]:
import pandas as pd 
import folium
from datetime import datetime,date

In [3]:
df = pd.read_parquet('crime.parquet')
df.shape

(7631663, 9)

In [4]:
df.head()

,unique_key,fh,primary_type,location_description,arrest,community_area,domestic,latitude,longitude
0,12332339,2021-04-05,ARSON,RESIDENCE - GARAGE,False,55,False,41.649456,-87.543518
1,8371884,2011-11-26,ARSON,BARBERSHOP,False,55,False,41.649491,-87.539898
2,13831847,2025-05-10,ARSON,STREET,False,54,False,41.650916,-87.606794
3,2430753,2002-10-31,ARSON,RESIDENCE PORCH/HALLWAY,False,55,False,41.651166,-87.548393
4,3930771,2005-04-19,ARSON,RESIDENCE,False,55,False,41.652884,-87.544727


In [18]:
aux = df[df['primary_type']=='HOMICIDE'].sample(100).reset_index(drop=True)
aux.shape

(100, 9)

In [19]:

# Create a map centered on the mean coordinates of the data
center_lat = aux['latitude'].mean()
center_lon = aux['longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

# Add markers for each crime location
for idx, row in aux.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Type: {row['primary_type']}<br>Location: {row['location_description']}<br>Arrest: {row['arrest']}",
        tooltip=f"Crime: {row['primary_type']}"
    ).add_to(m)

# Display the map
m


In [20]:
df['primary_type'].value_counts(1).index.tolist()

['THEFT',
 'BATTERY',
 'CRIMINAL DAMAGE',
 'NARCOTICS',
 'ASSAULT',
 'OTHER OFFENSE',
 'BURGLARY',
 'MOTOR VEHICLE THEFT',
 'DECEPTIVE PRACTICE',
 'ROBBERY',
 'CRIMINAL TRESPASS',
 'WEAPONS VIOLATION',
 'PROSTITUTION',
 'OFFENSE INVOLVING CHILDREN',
 'PUBLIC PEACE VIOLATION',
 'SEX OFFENSE',
 'CRIM SEXUAL ASSAULT',
 'INTERFERENCE WITH PUBLIC OFFICER',
 'GAMBLING',
 'HOMICIDE',
 'LIQUOR LAW VIOLATION',
 'ARSON',
 'CRIMINAL SEXUAL ASSAULT',
 'KIDNAPPING',
 'STALKING',
 'INTIMIDATION',
 'CONCEALED CARRY LICENSE VIOLATION',
 'OBSCENITY',
 'PUBLIC INDECENCY',
 'NON-CRIMINAL',
 'OTHER NARCOTIC VIOLATION',
 'HUMAN TRAFFICKING',
 'NON - CRIMINAL',
 'RITUALISM',
 'NON-CRIMINAL (SUBJECT SPECIFIED)']

In [21]:
high_impact_crimes_top5 = [
    "HOMICIDE",
    "CRIMINAL SEXUAL ASSAULT",
    "KIDNAPPING",
    "ROBBERY",
    "HUMAN TRAFFICKING"
]

In [44]:
aux = df[df['primary_type'].isin(high_impact_crimes_top5)].reset_index(drop=True)
aux = aux.loc[aux['fh']>=date(2002,1,1)].reset_index(drop=True)
aux['week'] = aux['fh'].map(lambda x:x.strftime('%Y-%U'))
aux.shape


(316924, 10)

In [46]:
aux.head()

,unique_key,fh,primary_type,location_description,arrest,community_area,domestic,latitude,longitude,week
0,11940157,2020-01-03,CRIMINAL SEXUAL ASSAULT,RESIDENCE,False,55,False,41.651431,-87.527258,2020-00
1,13762174,2025-02-28,CRIMINAL SEXUAL ASSAULT,APARTMENT,False,54,False,41.651432,-87.617231,2025-08
2,12927476,2022-12-10,CRIMINAL SEXUAL ASSAULT,RESIDENCE,False,54,False,41.653791,-87.598894,2022-49
3,12822220,2022-09-01,CRIMINAL SEXUAL ASSAULT,RESIDENCE,False,55,True,41.655019,-87.549561,2022-35
4,13055987,2023-03-09,CRIMINAL SEXUAL ASSAULT,CHA PARKING LOT / GROUNDS,False,54,False,41.657322,-87.606521,2023-10


In [54]:
piv = aux.pivot_table(index=['week','community_area'],columns='primary_type',values='unique_key',aggfunc='count',fill_value=0).reset_index()

In [55]:
column_mapping = {
    'week': 'week',
    'community_area': 'community_area', 
    'CRIMINAL SEXUAL ASSAULT': 'criminal_sexual_assault',
    'HOMICIDE': 'homicide',
    'HUMAN TRAFFICKING': 'human_trafficking',
    'KIDNAPPING': 'kidnapping',
    'ROBBERY': 'robbery'
}


In [56]:
piv.rename(columns=column_mapping, inplace=True)

In [57]:
piv.to_parquet('piv.parquet')

In [58]:
piv.columns

Index(['week', 'community_area', 'criminal_sexual_assault', 'homicide',
       'human_trafficking', 'kidnapping', 'robbery'],
      dtype='object', name='primary_type')

In [59]:
aux.head()

,unique_key,fh,primary_type,location_description,arrest,community_area,domestic,latitude,longitude,week
0,11940157,2020-01-03,CRIMINAL SEXUAL ASSAULT,RESIDENCE,False,55,False,41.651431,-87.527258,2020-00
1,13762174,2025-02-28,CRIMINAL SEXUAL ASSAULT,APARTMENT,False,54,False,41.651432,-87.617231,2025-08
2,12927476,2022-12-10,CRIMINAL SEXUAL ASSAULT,RESIDENCE,False,54,False,41.653791,-87.598894,2022-49
3,12822220,2022-09-01,CRIMINAL SEXUAL ASSAULT,RESIDENCE,False,55,True,41.655019,-87.549561,2022-35
4,13055987,2023-03-09,CRIMINAL SEXUAL ASSAULT,CHA PARKING LOT / GROUNDS,False,54,False,41.657322,-87.606521,2023-10


In [60]:
df.head()

,unique_key,fh,primary_type,location_description,arrest,community_area,domestic,latitude,longitude
0,12332339,2021-04-05,ARSON,RESIDENCE - GARAGE,False,55,False,41.649456,-87.543518
1,8371884,2011-11-26,ARSON,BARBERSHOP,False,55,False,41.649491,-87.539898
2,13831847,2025-05-10,ARSON,STREET,False,54,False,41.650916,-87.606794
3,2430753,2002-10-31,ARSON,RESIDENCE PORCH/HALLWAY,False,55,False,41.651166,-87.548393
4,3930771,2005-04-19,ARSON,RESIDENCE,False,55,False,41.652884,-87.544727


In [99]:
X = df.copy()
X['week'] = X['fh'].map(lambda x:x.strftime('%Y-%U'))

In [62]:
X.shape

(7631663, 10)

In [100]:
X['is_domestic'] = X['domestic']*1 
X['is_end_of_year']  = (X['week'].map(lambda x:x[-2:]).astype(int)>=49)*1

In [102]:
l = []
for v in ['primary_type','is_domestic']:
    l.append(X.pivot_table(index=['week','community_area','is_end_of_year'],
              columns=v,values='unique_key',aggfunc='count',fill_value=0))

Xf = pd.concat(l,axis=1)

In [103]:
Xf.drop(columns=0,inplace=True)


In [104]:
Xf.head()

ARSON  ASSAULT  BATTERY  BURGLARY  \
week    community_area is_end_of_year                                      
2001-00 1              0                   0        0        0         0   
        3              0                   0        0        0         0   
        4              0                   0        0        0         0   
        6              0                   0        0        0         0   
        7              0                   0        0        0         0   

                                       CONCEALED CARRY LICENSE VIOLATION  \
week    community_area is_end_of_year                                      
2001-00 1              0                                               0   
        3              0                                               0   
        4              0                                               0   
        6              0                                               0   
        7              0                                               0   

                                       CRIM SEXUAL ASSAULT  CRIMINAL DAMAGE  \
week    community_area is_end_of_year                                         
2001-00 1              0                                 0                0   
        3              0                                 0                0   
        4              0                                 0                0   
        6              0                                 0                0   
        7              0                                 0                0   

                                       CRIMINAL SEXUAL ASSAULT  \
week    community_area is_end_of_year                            
2001-00 1              0                                     0   
        3              0                                     0   
        4              0                                     0   
        6              0                                     0   
        7              0                                     0   

                                       CRIMINAL TRESPASS  DECEPTIVE PRACTICE  \
week    community_area is_end_of_year                                          
2001-00 1              0                               0                   3   
        3              0                               0                   0   
        4              0                               0                   0   
        6              0                               0                   0   
        7              0                               0                   0   

                                       ...  PROSTITUTION  PUBLIC INDECENCY  \
week    community_area is_end_of_year  ...                                   
2001-00 1              0               ...             0                 0   
        3              0               ...             0                 0   
        4              0               ...             0                 0   
        6              0               ...             0                 0   
        7              0               ...             0                 0   

                                       PUBLIC PEACE VIOLATION  RITUALISM  \
week    community_area is_end_of_year                                      
2001-00 1              0                                    0          0   
        3              0                                    0          0   
        4              0                                    0          0   
        6              0                                    0          0   
        7              0                                    0          0   

                                       ROBBERY  SEX OFFENSE  STALKING  THEFT  \
week    community_area is_end_of_year                                          
2001-00 1              0                     0            0         0      2   
        3              0                     0            1         0      

In [105]:
# Create mapping dictionary for column names
column_mapping = {
    'ARSON': 'x_arson',
    'ASSAULT': 'x_assault', 
    'BATTERY': 'x_battery',
    'BURGLARY': 'x_burglary',
    'CONCEALED CARRY LICENSE VIOLATION': 'x_concealed_carry_license_violation',
    'CRIM SEXUAL ASSAULT': 'x_crim_sexual_assault',
    'CRIMINAL DAMAGE': 'x_criminal_damage',
    'CRIMINAL SEXUAL ASSAULT': 'x_criminal_sexual_assault',
    'CRIMINAL TRESPASS': 'x_criminal_trespass',
    'DECEPTIVE PRACTICE': 'x_deceptive_practice',
    'GAMBLING': 'x_gambling',
    'HOMICIDE': 'x_homicide',
    'HUMAN TRAFFICKING': 'x_human_trafficking',
    'INTERFERENCE WITH PUBLIC OFFICER': 'x_interference_with_public_officer',
    'INTIMIDATION': 'x_intimidation',
    'KIDNAPPING': 'x_kidnapping',
    'LIQUOR LAW VIOLATION': 'x_liquor_law_violation',
    'MOTOR VEHICLE THEFT': 'x_motor_vehicle_theft',
    'NARCOTICS': 'x_narcotics',
    'NON - CRIMINAL': 'x_non_criminal',
    'NON-CRIMINAL': 'x_non_criminal',
    'NON-CRIMINAL (SUBJECT SPECIFIED)': 'x_non_criminal_subject_specified',
    'OBSCENITY': 'x_obscenity',
    'OFFENSE INVOLVING CHILDREN': 'x_offense_involving_children',
    'OTHER NARCOTIC VIOLATION': 'x_other_narcotic_violation',
    'OTHER OFFENSE': 'x_other_offense',
    'PROSTITUTION': 'x_prostitution',
    'PUBLIC INDECENCY': 'x_public_indecency',
    'PUBLIC PEACE VIOLATION': 'x_public_peace_violation',
    'RITUALISM': 'x_ritualism',
    'ROBBERY': 'x_robbery',
    'SEX OFFENSE': 'x_sex_offense',
    'STALKING': 'x_stalking',
    'THEFT': 'x_theft',
    'WEAPONS VIOLATION': 'x_weapons_violation',
    1: 'x_is_domestic'
}

In [106]:
Xf.rename(columns=column_mapping,inplace=True)

In [107]:
Xf.drop(columns=['x_non_criminal'],inplace=True)

In [108]:
Xf.reset_index().to_parquet('predictors.parquet')

In [95]:
print(",\n".join([f'coalesce({v},0) as {v}' for v in Xf.filter(like='x_').columns]))

coalesce(x_arson,0) as x_arson,
coalesce(x_assault,0) as x_assault,
coalesce(x_battery,0) as x_battery,
coalesce(x_burglary,0) as x_burglary,
coalesce(x_concealed_carry_license_violation,0) as x_concealed_carry_license_violation,
coalesce(x_crim_sexual_assault,0) as x_crim_sexual_assault,
coalesce(x_criminal_damage,0) as x_criminal_damage,
coalesce(x_criminal_sexual_assault,0) as x_criminal_sexual_assault,
coalesce(x_criminal_trespass,0) as x_criminal_trespass,
coalesce(x_deceptive_practice,0) as x_deceptive_practice,
coalesce(x_gambling,0) as x_gambling,
coalesce(x_homicide,0) as x_homicide,
coalesce(x_human_trafficking,0) as x_human_trafficking,
coalesce(x_interference_with_public_officer,0) as x_interference_with_public_officer,
coalesce(x_intimidation,0) as x_intimidation,
coalesce(x_kidnapping,0) as x_kidnapping,
coalesce(x_liquor_law_violation,0) as x_liquor_law_violation,
coalesce(x_motor_vehicle_theft,0) as x_motor_vehicle_theft,
coalesce(x_narcotics,0) as x_narcotics,
coalesce

In [98]:
print("\n".join([v for v in Xf.filter(like='x_').columns]))

x_arson
x_assault
x_battery
x_burglary
x_concealed_carry_license_violation
x_crim_sexual_assault
x_criminal_damage
x_criminal_sexual_assault
x_criminal_trespass
x_deceptive_practice
x_gambling
x_homicide
x_human_trafficking
x_interference_with_public_officer
x_intimidation
x_kidnapping
x_liquor_law_violation
x_motor_vehicle_theft
x_narcotics
x_non_criminal_subject_specified
x_obscenity
x_offense_involving_children
x_other_narcotic_violation
x_other_offense
x_prostitution
x_public_indecency
x_public_peace_violation
x_ritualism
x_robbery
x_sex_offense
x_stalking
x_theft
x_weapons_violation
x_is_domestic
